# Project 15 - RSNA Pneumonia Detection - Exploratory Data Analysis

**Dataset:** RSNA Pneumonia Detection Challenge (Kaggle, 2018)
**Task:** Object detection on frontal chest X-rays (DICOM); localise pneumonia opacities with bounding boxes.
**Author:** Sandeep Grover - Liora MLE Programme, Cohort 6973

Phase 1 EDA notebook - skeleton only. Cells are not executed in scaffolding. Run in main session after `kaggle competitions download` is complete (see `data/README.md`).

## 1. Imports and configuration

Pneumonia EDA on a DICOM-image dataset needs a few non-default libraries. `pydicom` reads DICOM files, OpenCV applies CLAHE pre-processing (Contrast Limited Adaptive Histogram Equalisation), and matplotlib plots images and overlays. Pandas handles the bounding-box CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pydicom
import cv2

DATA_DIR = Path('../data')
TRAIN_IMG_DIR = DATA_DIR / 'stage_2_train_images'
TEST_IMG_DIR = DATA_DIR / 'stage_2_test_images'
LABELS_CSV = DATA_DIR / 'stage_2_train_labels.csv'
CLASS_INFO_CSV = DATA_DIR / 'stage_2_detailed_class_info.csv'

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 8)

## 2. Load labels and class metadata

The bounding-box labels live in `stage_2_train_labels.csv` with one row per box. Patients can have multiple rows. The detailed class info adds a three-class split: `Normal`, `No Lung Opacity / Not Normal`, `Lung Opacity`. The middle class is critical and easy to miss (these are abnormal X-rays without pneumonia, e.g. cardiomegaly, atelectasis, effusion). A binary opacity-vs-rest model trained without seeing this distinction will misbehave clinically.

In [ ]:
labels = pd.read_csv(LABELS_CSV)
class_info = pd.read_csv(CLASS_INFO_CSV)
print('labels shape:', labels.shape)
print('class_info shape:', class_info.shape)
labels.head()

In [ ]:
labels.dtypes

In [ ]:
labels.describe(include='all')

In [ ]:
labels.isna().sum()

## 3. Target distribution and class balance

Two views: at the bounding-box row level (raw `Target` column) and at the patient level (any positive box per patient). The latter is what matters for model framing. Expect roughly 22-30 percent positive at patient level.

In [ ]:
row_target_counts = labels['Target'].value_counts()
patient_pos = labels.groupby('patientId')['Target'].max()
patient_target_counts = patient_pos.value_counts()
print('Row-level target counts:')
print(row_target_counts)
print('\nPatient-level positive rate:')
print(patient_target_counts)
print('Patient-level positive fraction:', patient_target_counts.get(1, 0) / patient_target_counts.sum())

In [ ]:
merged = labels.merge(class_info, on='patientId', how='left')
class_counts = merged.drop_duplicates('patientId')['class'].value_counts()
class_counts.plot(kind='bar')
plt.title('Patient-level class distribution')
plt.ylabel('Patients')
plt.tight_layout()

## 4. Bounding-box geometry

Inspect width, height, area, and centroid distributions. Pneumonia opacities are typically large (often a quarter of the image area) and centred mid-thorax. This shapes the choice of anchor scales and aspect ratios for a single-stage detector like RetinaNet, and the input size for YOLOv8.

In [ ]:
boxes = labels[labels['Target'] == 1].copy()
boxes['cx'] = boxes['x'] + boxes['width'] / 2
boxes['cy'] = boxes['y'] + boxes['height'] / 2
boxes['area'] = boxes['width'] * boxes['height']
boxes['aspect'] = boxes['width'] / boxes['height']
boxes[['width', 'height', 'area', 'aspect', 'cx', 'cy']].describe()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))
ax[0].hist(boxes['width'], bins=40); ax[0].set_title('Box width (px)')
ax[1].hist(boxes['height'], bins=40); ax[1].set_title('Box height (px)')
ax[2].hist(boxes['aspect'], bins=40); ax[2].set_title('Aspect ratio (w/h)')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(boxes['cx'], boxes['cy'], s=4, alpha=0.4)
plt.gca().invert_yaxis()
plt.title('Opacity-box centroids (image coords)')
plt.xlabel('x (px)')
plt.ylabel('y (px)')

## 5. Multi-box patients

How many positive patients have more than one box? Models that emit one detection per image will under-recall these cases.

In [ ]:
boxes_per_patient = boxes.groupby('patientId').size()
boxes_per_patient.value_counts().sort_index()

## 6. DICOM headers and visual sanity check

Read a positive and a negative case, plot pixel array with bounding-box overlay, and inspect a few DICOM tags (PatientAge, PatientSex, ViewPosition). These tags carry potential bias signals that the manuscript discusses (e.g. AP vs PA view position predicts disease in some sites because sicker patients get portable AP films).

In [ ]:
def load_dicom(patient_id, img_dir=TRAIN_IMG_DIR):
    ds = pydicom.dcmread(img_dir / f'{patient_id}.dcm')
    return ds

def show_with_boxes(patient_id, label_df=labels):
    ds = load_dicom(patient_id)
    img = ds.pixel_array
    rows = label_df[label_df['patientId'] == patient_id]
    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.imshow(img, cmap='gray')
    for _, r in rows.iterrows():
        if r['Target'] == 1:
            rect = patches.Rectangle((r['x'], r['y']), r['width'], r['height'],
                                     linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
    ax.set_title(f'{patient_id}  view={getattr(ds, "ViewPosition", "?")}  age={getattr(ds, "PatientAge", "?")}  sex={getattr(ds, "PatientSex", "?")}')
    return fig

pos_id = boxes['patientId'].iloc[0]
neg_id = labels[labels['Target'] == 0]['patientId'].iloc[0]
show_with_boxes(pos_id);
show_with_boxes(neg_id);

## 7. CLAHE pre-processing preview

CLAHE is the standard contrast-enhancement step for chest X-rays in CV pipelines (CheXNet conventions, MICCAI tutorials). It boosts local contrast in the lung parenchyma without globally over-saturating. Preview pre and post.

In [ ]:
ds = load_dicom(pos_id)
img = ds.pixel_array.astype(np.uint8)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
img_clahe = clahe.apply(img)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('Raw DICOM')
ax[1].imshow(img_clahe, cmap='gray'); ax[1].set_title('CLAHE')
for a in ax:
    a.axis('off')

## 8. Demographic and view-position summary

Aggregate DICOM metadata across a sample of patients to inspect age distribution, sex split, and AP-vs-PA view balance. Skip in scaffolding (slow) - run on a 1k random sample in the main session.

In [ ]:
# sample = labels['patientId'].drop_duplicates().sample(1000, random_state=0)
# meta = []
# for pid in sample:
#     ds = load_dicom(pid)
#     meta.append({
#         'patientId': pid,
#         'age': getattr(ds, 'PatientAge', None),
#         'sex': getattr(ds, 'PatientSex', None),
#         'view': getattr(ds, 'ViewPosition', None),
#     })
# meta_df = pd.DataFrame(meta)
# meta_df.describe(include='all')

## 9. Notes for modelling

- Class imbalance: roughly 22-30 percent positive at patient level. Use focal loss (RetinaNet baseline) or class-weighted sampling (YOLOv8 advanced). Both are documented in `src/`.
- Multi-box images: keep all boxes during training; do not collapse to a single per-image label.
- Anchor design (RetinaNet): box areas span roughly 80 to 700 pixels per side at 1024 res. Default RetinaNet anchors are a reasonable starting point; consider re-clustering for tighter recall.
- Pre-processing: CLAHE plus uint8 conversion plus resize to 512 or 640. Heavy augmentation in advanced model: affine, brightness/contrast, mixup. Avoid vertical flips (anatomy-breaking).
- Validation: stratified split on patient-level Target with class info preserved. Never split inside a patient.
- External-validation caveat: the dataset is curated from NIH ChestX-ray (Wang 2017). Site, scanner, and demographic coverage differs from arbitrary deployment hospitals; cross-site generalisation should be tested before clinical use (see Zech 2018 and related references).